### Transform Customer Data
- Remove records with NULL Customer_id
- Remove exact duplicate reords
- Remove duplicates based on the created_timesatmp
- CAST the columns to correct datatypes
- Write transformed data to the silver schema

#### 1. Remove records with NULL Customer_id

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.table.html
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html


In [0]:
from pyspark.sql.functions import col, max, to_date, to_timestamp

df = spark.read.table("gizmobox.bronze.py_customers").where("customer_id IS NOT NULL").distinct()
max_ts_df = df.groupBy(col("customer_id")).agg(max(col("created_timestamp")).alias("created_timestamp"))
final_df = df.join(max_ts_df,["customer_id","created_timestamp"],"inner")
output_df = final_df.select(
    col("customer_id"),
    col("customer_name"),
    to_date(col("date_of_birth")),
    col("email"),
    col("member_since"),
    col("telephone"),
    to_timestamp(col("created_timestamp"))
)
display(output_df)


In [0]:
df = spark.sql("""
SELECT
    DiSTINCT *
FROM gizmobox.bronze.py_customers
WHERE customer_id IS NOT NULL;        
          """)
display(df)

In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
filter_df = df.filter("customer_id IS NOT NULL")
display(filter_df)

In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
filter_df = df.filter(df.customer_id.isNotNull())
display(filter_df)

#### 2. Remove exact duplicate reords
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop_duplicates.html
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.distinct.html


In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
distinct_df = df.filter(df.customer_id.isNotNull()).distinct()
display(distinct_df)

In [0]:
df = spark.read.table("gizmobox.bronze.py_customers")
distinct_df = df.filter(df.customer_id.isNotNull()).dropDuplicates()
display(distinct_df)

#### 3. Remove duplicates based on the created_timesatmp
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, to_timestamp

max_ts_df = distinct_df.groupBy(col("customer_id")).agg(
                    F.max(col("created_timestamp")).alias("max_created_timestamp")
                )
final_df = distinct_df.join(max_ts_df, (distinct_df.customer_id == max_ts_df.customer_id) & (distinct_df.created_timestamp == max_ts_df.max_created_timestamp), "inner").select(df['*'])

display(final_df)

####4. CAST the columns to correct datatypes

In [0]:

casted_df = final_df.select(
    final_df.customer_id,
    final_df.customer_name,
    final_df.date_of_birth.cast("date"),
    final_df.email,
    final_df.member_since.cast("date"),
    final_df.telephone,
    final_df.created_timestamp.cast("timestamp")
)
display(casted_df)

#### 5. Write transformed data to the silver schema

In [0]:
casted_df.writeTo("gizmobox.silver.py_customers").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.customers;